In [1]:
import pathlib as plib
import numpy as np
import twixtools as tt

import plotly.graph_objects as go
import plotly.subplots as psub
import plotly.colors as plc

## Load data
we load the raw data file using twixtools "high-level" function map_twix, such that we dont have to take care of all the sorting and extraction of flags ourselves

In [2]:
path = plib.Path("/data/pt_02262/data/TH_bids/source/41006.a1/20231123/raw").absolute()
in_file = path.joinpath("meas_MID00034_semc_js_res0p6_etl7_esp9_grappa3_slice52_20231123-212035").with_suffix(".dat")
path_figs = plib.Path(__name__).parent.absolute().joinpath("figs")
twix = tt.map_twix(in_file.as_posix())[-1]

Software version: VD/VE (!?)

Scan  0


3.85GB [00:18, 225MB/s]                            


We now have a twix dictionary object.
Since its a Grappa scan this is supposed to have 3 separate data arrays:
- a noise scan -> essentially just opening an ADC without any spin manipulation. Thus its just a bunch of samples of thermal noise (and some other effects besides "thermal")
- an image scan -> this is the accelerated / sub-sampled image data, it will contain every n-th line with n being the acceleration set in the sequence parameters
- a reference scan -> this contains the central region of the k-space which is fully sampled. The number of lines are also set in the sequence parameters via -> GRAPPA / PAT number of reference lines
 
The image scan will have some lines that it shares with the central region. I never checked if those lines are actually acquired twice (i don't think since it would increase the time needed) or if they are just copied to both arrays. What GRAPPA does to fill the missing lines in the image is try to use kernels that interpolate neighboring lines and coils for the missing lines and solve this interpolation for the central region where we have those missing lines in the ref scan. Those kernels are then extended to the whole image.

We will first do some rearranging. you can check the dimensions of the variables:
- Noise [num-coils, num-samples]
- Image [echoes, slices, phase - encodes, coils, readout]
- ref-scan [echoes, slices, phase - encodes, coils, readout]

The readout is oversampled, hence it has twice as many samples as the set base resolution

In [3]:
noise = np.squeeze(np.asarray(twix["noise"][:]))
# remove oversampling
twix["image"].flags["remove_os"] = True
twix["refscan"].flags["remove_os"] = True
# extract numpy arrays
img = np.squeeze(np.asarray(twix["image"][:]))
refscan = np.squeeze(np.asarray(twix["refscan"][:]))
# reorder to [read, phase, slice, channel, echoes]
img = np.permute_dims(img, [-1, 2, 1, -2, 0])
refscan = np.permute_dims(refscan, [-1, 2, 1, -2, 0])

We want to build a combined k-space including the central region and the sub-sampled outer lines

In [4]:
# find indices of ref-scan phase encodes
pe_ind = np.nonzero(np.abs(refscan[refscan.shape[0]//2, :, 0, 0, 0]))[0]
print(f"Assumed number of ref-lines: {pe_ind.shape[0]}")
# sort into the image
k_combined = img.copy()
k_combined[:, pe_ind] = refscan[:, pe_ind]
# to take a look at the naive fft recon of the undersampled data:
img_fft = np.fft.fftshift(
    np.fft.fft2(
        np.fft.ifftshift(
            k_combined[:, :, k_combined.shape[2] // 2, 0, 0], axes=(0, 1)
        ), 
        axes=(0, 1)
    ), 
    axes=(0, 1)
)

Assumed number of ref-lines: 42


In [27]:
fig = psub.make_subplots(rows=1, cols=4)

for i, d in enumerate([img, refscan, img_combined, img_fft]):
    d = np.log(np.abs(d[:, :, d.shape[2] // 2, 0, 0])) if i < 3 else np.abs(d) 
    fig.add_trace(
        go.Heatmap(
            z=d, showscale=False, showlegend=False,
            colorscale="Inferno"
        ),
        row=1, col=1+i
    )
    x = fig.data[-1].xaxis
    fig.update_xaxes(visible=False, row=1, col=1+i)
    fig.update_yaxes(visible=False, scaleanchor=x, row=1, col=1+i)
fn = path_figs.joinpath("k-space").with_suffix(".html")
print(f"write file: {fn}")
fig.write_html(fn)

/tmp/ipykernel_25171/4219069863.py:4: RuntimeWarning:

divide by zero encountered in log



write file: /data/pt_np-jschmidt/code/PyMRItools/jupy_nbs/jupy_nbs/figs/k-space.html


This combined k-space will be, after some tweaks, the input for the AC-LORAKS recon


## Denoising
For the denoising we need the noise adjustment scan. Additionally we will use this scan also to do the pre-whitening
Lets take a look at the coil correlations and the coil noise stats



Pre-whitening.
We want to end up with "uncorrelated" iid gaussian noise coil noise stats.

In [36]:
# calcualte covariance matrix
cov = np.cov(noise)
# get pseudo inverse
psi_l = np.linalg.cholesky(cov)
psi_l = np.linalg.inv(psi_l)

# we can check how well the pre-whitening worked 
noise_prew = np.einsum("ij, ki -> kj", noise, psi_l)

In [44]:
fig = psub.make_subplots(
    rows=2, cols=3,
    row_titles=["Noise raw", "Noise Prew."],
    column_titles=["Correlation", "Real Hist.", "Imag Hist."]
)

cmap = plc.sample_colorscale("Inferno", noise.shape[0], 0.1, 0.9)
for a, n in enumerate([noise, noise_prew]):
    coil_corr = np.corrcoef(n)    
    fig.add_trace(
        go.Heatmap(
            z=np.abs(coil_corr), showscale=False, showlegend=False, colorscale="Inferno"
        ),
        col=1, row=1+a
    )
    for b, d in enumerate(n): 
        for c, f in enumerate([np.real, np.imag]):
            hist, bins = np.histogram(f(d))
            bins = bins[1:] - np.diff(bins) / 2
            fig.add_trace(
                go.Scatter(y=hist, x=bins, mode="lines", line=dict(color=cmap[b]), showlegend=False),
                col=2+c, row=1+a
            )
fn = path_figs.joinpath("noise").with_suffix(".html")
print(f"write file: {fn}")
fig.write_html(fn)

write file: /data/pt_np-jschmidt/code/PyMRItools/jupy_nbs/jupy_nbs/figs/noise.html


In [46]:
# We use the same transformation to pre-whiten the data
k_comb_prew = np.einsum(
    "rpsce, wc -> rpswe", k_combined, psi_l
)

We now have the data and the noise pre-whitened and brought into the format needed for LORAKS.
We port and ship them to torch tensors which are used by my denoising package, which also assumes more than one noise scans, hence unsqueeze the dimension if theres only one


In [47]:
import torch

path_out = plib.Path(__name__).parent.absolute()
k_cp = torch.from_numpy(k_comb_prew) 
noise_p = torch.from_numpy(noise_prew)
if noise_p.shape.__len__() < 3:
    noise_p = noise_p.unsqueeze(0)

for i, d in enumerate([k_cp, noise_p]):
    name = path_out.joinpath(["k_space", "noise"][i]).with_suffix(".pt")
    print(f"write file: {name}")
    torch.save(d, name)

write file: /data/pt_np-jschmidt/code/PyMRItools/jupy_nbs/jupy_nbs/k_space.pt
write file: /data/pt_np-jschmidt/code/PyMRItools/jupy_nbs/jupy_nbs/noise.pt
